# LeetCode 206 — Reverse Linked List bằng Python

**Link đề:** https://leetcode.com/problems/reverse-linked-list/
**Độ khó:** Easy · **Chủ đề:** Linked List, Recursion

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def reverseList(self, head: ListNode | None) -> ListNode | None:
        ...
```

## Mục tiêu học
1. Nắm chắc linked list: node, `val`, `next`, `head`, `None`, và việc "gán biến không sao chép node".
2. Hiểu "đảo danh sách" = **đảo chiều mọi mũi tên `next`**, không phải đổi chỗ giá trị.
3. Thành thạo kỹ thuật **ba con trỏ** `prev / cur / nxt` — một trong những mẫu code quan trọng nhất về linked list.
4. Viết được cả bản **lặp** (iterative) và **đệ quy** (recursive) như follow-up của đề.
5. Trace từng bước, chứng minh bằng invariant, so sánh 4 cách theo Time/Space.

## 1. Đề bài

Cho `head` của một singly linked list, hãy **đảo ngược** danh sách và trả về head mới.

| Input | Output |
|---|---|
| `1 → 2 → 3 → 4 → 5` | `5 → 4 → 3 → 2 → 1` |
| `1 → 2` | `2 → 1` |
| rỗng | rỗng |

**Constraints:** số node trong `[0, 5000]`; `-5000 <= Node.val <= 5000`.

**Follow up:** Có thể đảo bằng vòng lặp hoặc đệ quy. Hãy cài đặt cả hai.

Hình dung "đảo" ở mức mũi tên:

```text
trước:  1 -> 2 -> 3 -> None
sau:    None <- 1 <- 2 <- 3        (head mới là 3)
```
Node `1` (head cũ) trở thành node cuối, nên `1.next` phải là `None`.

## 2. Nền tảng: `ListNode` và hàm phụ

Một node có `val` (giá trị) và `next` (tham chiếu tới node sau hoặc `None`). LeetCode cung cấp sẵn class này khi chấm; trong notebook ta tự định nghĩa.

Hàm phụ (chỉ để học/test): `build_list` tạo linked list từ list Python, `to_list` thu lại các giá trị, `show` in dạng mũi tên.

In [1]:
from typing import Optional

class ListNode:
    def __init__(self, val: int = 0, next: "Optional[ListNode]" = None):
        self.val = val
        self.next = next

def build_list(values: list[int]) -> Optional[ListNode]:
    head = None
    for v in reversed(values):        # dựng từ cuối lên: node mới trỏ tới head cũ
        head = ListNode(v, head)
    return head

def to_list(head: Optional[ListNode]) -> list[int]:
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out

def show(head: Optional[ListNode]) -> str:
    return " -> ".join(map(str, to_list(head))) + " -> None" if head else "None"

h = build_list([1, 2, 3, 4, 5])
print(show(h))
print("head.val =", h.val, "| head.next.val =", h.next.val)

1 -> 2 -> 3 -> 4 -> 5 -> None
head.val = 1 | head.next.val = 2


### 2.1. Biến chỉ là "tên dán lên node", không phải bản sao

`cur = head` không tạo node mới — `cur` và `head` cùng chỉ vào một node. Đổi `cur.next` là đổi luôn `head.next`. Đây là lý do khi đảo, ta phải **lưu node kế tiếp trước** khi sửa `next`, nếu không sẽ mất đường đi tiếp.

`is` kiểm tra hai biến có cùng chỉ **một đối tượng** không.

In [2]:
head = build_list([1, 2, 3])
cur = head
print("cur is head?", cur is head)
cur.next = None                  # cắt mũi tên của node 1
print("Sau khi cắt qua cur, head còn:", show(head), "-> node 2, 3 bị 'mất dấu'!")

cur is head? True
Sau khi cắt qua cur, head còn: 1 -> None -> node 2, 3 bị 'mất dấu'!


### 2.2. Gán nhiều biến cùng lúc (tuple assignment)

`a, b = b, a` tính **toàn bộ vế phải trước** rồi mới gán. Nhờ vậy hoán đổi không cần biến tạm. Ta sẽ gặp lại cú pháp này ở dạng rút gọn của thuật toán.

In [3]:
a, b_ = 1, 2
a, b_ = b_, a
print("a =", a, "| b_ =", b_)
x, y, z = 10, 20, 30
x, y, z = y, z, x
print(x, y, z)

a = 2 | b_ = 1
20 30 10


## 3. Cách 1 — Baseline: chép giá trị ra list Python rồi dựng lại

Ý tưởng tự nhiên: "list Python đảo được dễ dàng bằng `[::-1]`".
1. Duyệt linked list, gom giá trị vào list `vals`.
2. Dựng linked list **mới** từ `vals[::-1]`.

Đúng kết quả, nhưng tạo `n` node mới và list phụ → bộ nhớ `O(n)`; và không thực sự "đảo" danh sách gốc. Phỏng vấn thường yêu cầu đảo **tại chỗ**.

In [4]:
def reverse_copy(head: Optional[ListNode]) -> Optional[ListNode]:
    vals = to_list(head)
    return build_list(vals[::-1])

print(show(reverse_copy(build_list([1, 2, 3, 4, 5]))))
print(show(reverse_copy(None)))

5 -> 4 -> 3 -> 2 -> 1 -> None
None


## 4. Cách 2 — Dùng stack chứa các node (đảo tại chỗ nhưng tốn bộ nhớ)

**Stack** (ngăn xếp) là cấu trúc "vào sau ra trước" (LIFO). Trong Python dùng list: `append` để đẩy vào, `pop()` để lấy phần tử **cuối**.

Đẩy mọi node vào stack, rồi `pop` lần lượt và nối lại: node lấy ra đầu tiên là node cuối cũ = head mới. Không tạo node mới, nhưng stack vẫn tốn `O(n)`.

In [5]:
stack = []
stack.append(1); stack.append(2); stack.append(3)
print("stack:", stack, "| pop ->", stack.pop(), "| còn:", stack)

def reverse_stack(head: Optional[ListNode]) -> Optional[ListNode]:
    stack = []
    while head:
        stack.append(head)
        head = head.next
    if not stack:
        return None
    new_head = stack.pop()
    tail = new_head
    while stack:
        tail.next = stack.pop()
        tail = tail.next
    tail.next = None               # node cuối mới (head cũ) phải trỏ None
    return new_head

print(show(reverse_stack(build_list([1, 2, 3, 4, 5]))))

stack: [1, 2] | pop -> 3 | còn: [1, 2]
5 -> 4 -> 3 -> 2 -> 1 -> None


## 5. Nút thắt & insight

Nút thắt: cách 1, 2 cần `O(n)` bộ nhớ phụ chỉ để "nhớ đường quay lại".

**Insight:** ta không cần nhớ cả danh sách. Khi đứng ở node `cur`, chỉ cần biết:
- `prev`: node đứng **trước** `cur` — nơi `cur.next` phải trỏ về sau khi đảo;
- `nxt`: node đứng **sau** `cur` — để không mất đường đi tiếp khi ta ghi đè `cur.next`.

Mỗi bước làm **một** việc: đảo đúng một mũi tên `cur.next = prev`, rồi cả ba con trỏ dịch sang phải một node. Phần bên trái `prev` đã đảo xong, phần từ `cur` trở đi chưa đảo.

```text
 đã đảo        đang xử lý   chưa đảo
None <- 1 <- 2    3     ->  4 -> 5 -> None
             prev cur       nxt
```

## 6. Cách 3 — Lời giải chính: ba con trỏ (iterative)

**Đoạn 1 — khởi tạo**
```python
prev = None        # head cũ sẽ trỏ về None
cur = head
```

**Đoạn 2 — còn node chưa đảo thì lặp**
```python
while cur is not None:
```

**Đoạn 3 — bốn dòng cốt lõi (thứ tự rất quan trọng)**
```python
    nxt = cur.next     # (1) lưu đường đi tiếp
    cur.next = prev    # (2) đảo mũi tên
    prev = cur         # (3) prev tiến lên
    cur = nxt          # (4) cur tiến lên
```

**Đoạn 4 — trả head mới**
```python
return prev            # khi cur là None, prev là node cuối cũ = head mới
```

Mẹo nhớ thứ tự: "**lưu – đảo – tiến – tiến**". Dòng sau dùng giá trị dòng trước vừa giải phóng.

In [6]:
def reverse_iterative(head: Optional[ListNode]) -> Optional[ListNode]:
    prev = None
    cur = head
    while cur is not None:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev

for vals in [[1, 2, 3, 4, 5], [1, 2], [], [42]]:
    print(f"{vals!s:<16} -> {show(reverse_iterative(build_list(vals)))}")

[1, 2, 3, 4, 5]  -> 5 -> 4 -> 3 -> 2 -> 1 -> None
[1, 2]           -> 2 -> 1 -> None
[]               -> None
[42]             -> 42 -> None


## 7. TRACE từng bước

### 7.1. Bảng trace `1 → 2 → 3`

| Vòng | `cur` | `nxt = cur.next` | Đảo `cur.next = prev` | `prev` sau | `cur` sau | Phần đã đảo (bắt đầu từ `prev`) |
|---:|:-:|:-:|---|:-:|:-:|---|
| 1 | 1 | 2 | `1 → None` | 1 | 2 | `1 → None` |
| 2 | 2 | 3 | `2 → 1` | 2 | 3 | `2 → 1 → None` |
| 3 | 3 | None | `3 → 2` | 3 | None | `3 → 2 → 1 → None` |
| — | None | | dừng | | | trả `prev = 3` |

In [7]:
def trace_reverse(head: Optional[ListNode]) -> Optional[ListNode]:
    prev, cur = None, head
    step = 1
    print(f"Ban đầu: chưa đảo = {show(cur)}")
    while cur is not None:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
        print(f"Vòng {step}: đảo node {prev.val}; đã đảo = {show(prev):<28} | chưa đảo = {show(cur)}")
        step += 1
    print("Kết quả: head mới =", prev.val if prev else None)
    return prev

trace_reverse(build_list([1, 2, 3, 4, 5]))

Ban đầu: chưa đảo = 1 -> 2 -> 3 -> 4 -> 5 -> None
Vòng 1: đảo node 1; đã đảo = 1 -> None                    | chưa đảo = 2 -> 3 -> 4 -> 5 -> None
Vòng 2: đảo node 2; đã đảo = 2 -> 1 -> None               | chưa đảo = 3 -> 4 -> 5 -> None
Vòng 3: đảo node 3; đã đảo = 3 -> 2 -> 1 -> None          | chưa đảo = 4 -> 5 -> None
Vòng 4: đảo node 4; đã đảo = 4 -> 3 -> 2 -> 1 -> None     | chưa đảo = 5 -> None
Vòng 5: đảo node 5; đã đảo = 5 -> 4 -> 3 -> 2 -> 1 -> None | chưa đảo = None
Kết quả: head mới = 5


### 7.2. Trace biên — danh sách rỗng và một node

- Rỗng: `cur = None` → vòng lặp không chạy → trả `prev = None`. Đúng.
- Một node: một vòng, `1.next = None`, trả node 1. Đúng.

In [8]:
trace_reverse(None)
print()
trace_reverse(build_list([7]))

Ban đầu: chưa đảo = None
Kết quả: head mới = None

Ban đầu: chưa đảo = 7 -> None
Vòng 1: đảo node 7; đã đảo = 7 -> None                    | chưa đảo = None
Kết quả: head mới = 7


### 7.3. Dạng rút gọn bằng tuple assignment

```python
while cur:
    cur.next, prev, cur = prev, cur, cur.next
```

Vế phải `(prev, cur, cur.next)` được tính **hết** trước, nên giá trị `cur.next` cũ đã được giữ lại. Ngắn nhưng khó đọc hơn — chỉ dùng khi đã hiểu bản 4 dòng.

⚠️ Thứ tự **vế trái** cũng quan trọng: Python gán từ trái sang phải. Nếu viết `prev, cur, cur.next = cur, cur.next, prev` thì `cur` bị đổi trước, rồi `cur.next = prev` lại gán vào **node mới** → sai.

In [9]:
def reverse_tuple(head):
    prev, cur = None, head
    while cur:
        cur.next, prev, cur = prev, cur, cur.next
    return prev

def reverse_tuple_wrong_order(head):
    prev, cur = None, head
    while cur:
        prev, cur, cur.next = cur, cur.next, prev    # thứ tự gán sai
    return prev

print("Đúng:", show(reverse_tuple(build_list([1, 2, 3, 4]))))
try:
    print("Sai thứ tự:", show(reverse_tuple_wrong_order(build_list([1, 2, 3, 4]))))
except AttributeError as e:
    print("Sai thứ tự -> lỗi:", e)

Đúng: 4 -> 3 -> 2 -> 1 -> None
Sai thứ tự -> lỗi: 'NoneType' object has no attribute 'next' and no __dict__ for setting new attributes


Quan sát output: bản sai không đảo được mà còn làm hỏng danh sách (hoặc báo lỗi vì gán `.next` lên `None`). Bài học: với tuple assignment có thuộc tính (`cur.next`), **đặt `cur.next` ở đầu vế trái**.

## 8. Cách 4 — Đệ quy (follow-up)

Tư duy "tin tưởng lời gọi con":
1. **Cơ sở:** danh sách rỗng hoặc 1 node → đã đảo sẵn, trả `head`.
2. Đảo phần còn lại: `new_head = reverse(head.next)`. Sau lời gọi này, `head.next` (node thứ 2) đã thành **node cuối** của phần đảo.
3. Nối `head` vào sau node đó: `head.next.next = head`.
4. Cắt mũi tên cũ của `head`: `head.next = None` (vì `head` giờ là node cuối).
5. Trả `new_head` (head mới luôn là node cuối cũ, được truyền ngược lên mọi tầng).

```text
reverse(1 -> 2 -> 3):
  reverse(2 -> 3) trả 3 -> 2 -> None, nhưng 1 vẫn trỏ tới 2
  1.next.next = 1   =>  3 -> 2 -> 1
  1.next = None     =>  3 -> 2 -> 1 -> None
```

In [10]:
def reverse_recursive(head: Optional[ListNode]) -> Optional[ListNode]:
    if head is None or head.next is None:
        return head
    new_head = reverse_recursive(head.next)
    head.next.next = head
    head.next = None
    return new_head

for vals in [[1, 2, 3, 4, 5], [1, 2], [], [42]]:
    print(f"{vals!s:<16} -> {show(reverse_recursive(build_list(vals)))}")

[1, 2, 3, 4, 5]  -> 5 -> 4 -> 3 -> 2 -> 1 -> None
[1, 2]           -> 2 -> 1 -> None
[]               -> None
[42]             -> 42 -> None


In [11]:
def trace_recursive(head, depth=0):
    pad = "  " * depth
    if head is None or head.next is None:
        print(pad + f"cơ sở: node {head.val if head else None} -> trả chính nó")
        return head
    print(pad + f"gọi đảo phần sau node {head.val}")
    new_head = trace_recursive(head.next, depth + 1)
    head.next.next = head
    head.next = None
    print(pad + f"nối {head.val} vào sau; phần đã đảo = {show(new_head)}")
    return new_head

trace_recursive(build_list([1, 2, 3, 4]))

gọi đảo phần sau node 1
  gọi đảo phần sau node 2
    gọi đảo phần sau node 3
      cơ sở: node 4 -> trả chính nó
    nối 3 vào sau; phần đã đảo = 4 -> 3 -> None
  nối 2 vào sau; phần đã đảo = 4 -> 3 -> 2 -> None
nối 1 vào sau; phần đã đảo = 4 -> 3 -> 2 -> 1 -> None


### 8.1. Giới hạn đệ quy

Mỗi lời gọi chiếm một khung trên **call stack** → bộ nhớ `O(n)`. Python mặc định giới hạn khoảng 1000 tầng; với 5000 node bản đệ quy có thể báo `RecursionError` nếu môi trường không nâng giới hạn (LeetCode có nâng, nhưng đừng dựa vào). Bản lặp không có vấn đề này.

In [12]:
import sys
print("Giới hạn mặc định:", sys.getrecursionlimit())
long_list = build_list(list(range(3000)))
try:
    reverse_recursive(long_list)
    print("3000 node: chạy được")
except RecursionError as e:
    print("3000 node: RecursionError ->", e)
print("Bản lặp 5000 node, 3 giá trị đầu sau khi đảo:", to_list(reverse_iterative(build_list(list(range(5000)))))[:3])

Giới hạn mặc định: 1000
3000 node: RecursionError -> maximum recursion depth exceeded
Bản lặp 5000 node, 3 giá trị đầu sau khi đảo: [4999, 4998, 4997]


## 9. Code nộp LeetCode

Bản lặp (khuyến nghị):

In [13]:
class Solution:
    def reverseList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        prev = None
        cur = head
        while cur:
            nxt = cur.next
            cur.next = prev
            prev = cur
            cur = nxt
        return prev

print(show(Solution().reverseList(build_list([1, 2, 3, 4, 5]))))

5 -> 4 -> 3 -> 2 -> 1 -> None


Bản đệ quy (follow-up):

In [14]:
class SolutionRecursive:
    def reverseList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        if not head or not head.next:
            return head
        new_head = self.reverseList(head.next)
        head.next.next = head
        head.next = None
        return new_head

print(show(SolutionRecursive().reverseList(build_list([1, 2]))))

2 -> 1 -> None


## 10. Vì sao đúng? (invariant)

**Bản lặp** — ở đầu mỗi vòng:

> `prev` là head của danh sách gồm các node đã duyệt, **đã đảo đúng thứ tự ngược**; `cur` là head của phần **chưa đụng tới**, giữ nguyên thứ tự gốc. Hai phần rời nhau và hợp lại đủ `n` node.

- **Ban đầu:** `prev = None` (phần đã đảo rỗng), `cur = head` (toàn bộ chưa đụng) → đúng.
- **Một vòng:** lấy node `cur` ra khỏi đầu phần chưa đụng (nhờ đã lưu `nxt`), gắn nó lên **đầu** phần đã đảo (`cur.next = prev`), rồi cập nhật `prev = cur`, `cur = nxt` → invariant giữ nguyên, phần chưa đụng giảm 1 node.
- **Kết thúc:** `cur is None` → phần chưa đụng rỗng → `prev` là toàn bộ danh sách đã đảo.

**Bản đệ quy** — quy nạp theo độ dài: đúng với 0, 1 node; giả sử đúng với `n-1` node thì sau lời gọi con, phần `head.next...` đã đảo và `head.next` là node cuối; nối `head` vào sau nó và cắt `head.next` cho ta `n` node đảo đúng.

## 11. Time / Space — so sánh 4 cách

Gọi `n` là số node.

| Cách | Ý tưởng | Time | Space phụ | Sửa tại chỗ? |
|---|---|---|---|:---:|
| 1. Chép list rồi dựng lại | `vals[::-1]` | `O(n)` | `O(n)` | không |
| 2. Stack node | LIFO | `O(n)` | `O(n)` | có |
| 3. Ba con trỏ | lưu–đảo–tiến–tiến | `O(n)` | `O(1)` | có — **chuẩn** |
| 4. Đệ quy | tin lời gọi con | `O(n)` | `O(n)` call stack | có |

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
run = lambda vals: to_list(sol.reverseList(build_list(vals)))
check("ví dụ 1", run([1, 2, 3, 4, 5]), [5, 4, 3, 2, 1])
check("ví dụ 2", run([1, 2]), [2, 1])
check("ví dụ 3 rỗng", run([]), [])
check("một node", run([9]), [9])
check("giá trị âm & trùng", run([-5000, 0, 0, 5000]), [5000, 0, 0, -5000])
check("5000 node", run(list(range(5000)))[:3], [4999, 4998, 4997])

# Đảo hai lần phải quay về ban đầu
h = build_list([3, 1, 4, 1, 5])
check("đảo 2 lần", to_list(sol.reverseList(sol.reverseList(h))), [3, 1, 4, 1, 5])

# Node cuối mới (head cũ) phải trỏ None — tránh chu trình
h = build_list([1, 2, 3]); old_head = h
new = sol.reverseList(h)
check("head cũ thành đuôi", old_head.next is None, True)

PASS ví dụ 1: [5, 4, 3, 2, 1]
PASS ví dụ 2: [2, 1]
PASS ví dụ 3 rỗng: []
PASS một node: [9]
PASS giá trị âm & trùng: [5000, 0, 0, -5000]
PASS 5000 node: [4999, 4998, 4997]
PASS đảo 2 lần: [3, 1, 4, 1, 5]
PASS head cũ thành đuôi: True


In [16]:
import random
rng = random.Random(206)
methods = [reverse_copy, reverse_stack, reverse_iterative, reverse_tuple, reverse_recursive,
           Solution().reverseList, SolutionRecursive().reverseList]
for _ in range(2000):
    vals = [rng.randint(-5, 5) for _ in range(rng.randint(0, 15))]
    for m in methods:
        assert to_list(m(build_list(vals))) == vals[::-1], (m, vals)
print("7 cài đặt khớp vals[::-1] trên 2000 input ngẫu nhiên")

7 cài đặt khớp vals[::-1] trên 2000 input ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đảo mũi tên trước khi lưu `nxt`

```python
while cur:
    cur.next = prev       # mất đường đi tới node sau!
    prev = cur
    cur = cur.next        # cur.next lúc này là prev -> đi lùi
```

In [17]:
def reverse_bug_no_save(head):
    prev, cur = None, head
    steps = 0
    while cur and steps < 10:     # chặn 10 bước để không treo
        cur.next = prev
        prev = cur
        cur = cur.next
        steps += 1
    return prev

print("Bản lỗi [1,2,3,4]:", show(reverse_bug_no_save(build_list([1, 2, 3, 4]))), "(mất node 2,3,4)")

Bản lỗi [1,2,3,4]: 1 -> None (mất node 2,3,4)


### Lỗi 2 — Trả `cur` hoặc `head` thay vì `prev`

Sau vòng lặp `cur` là `None`; còn `head` cũ giờ là **đuôi** (chỉ còn 1 node `head → None`). Head mới là `prev`.

In [18]:
def reverse_bug_return_head(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next; cur.next = prev; prev = cur; cur = nxt
    return head

print("Trả head cũ:", show(reverse_bug_return_head(build_list([1, 2, 3]))), "(đúng phải là 3 -> 2 -> 1)")

Trả head cũ: 1 -> None (đúng phải là 3 -> 2 -> 1)


### Lỗi 3 — Đệ quy quên `head.next = None` → tạo chu trình

Thiếu dòng này, node 1 và node 2 trỏ nhau: `2 → 1 → 2 → 1 ...`. In danh sách sẽ chạy mãi. Cell dưới chỉ in tối đa 8 node để thấy chu trình.

In [19]:
def reverse_bug_cycle(head):
    if head is None or head.next is None:
        return head
    new_head = reverse_bug_cycle(head.next)
    head.next.next = head
    # quên: head.next = None
    return new_head

def first_k(head, k=8):
    out = []
    while head and len(out) < k:
        out.append(head.val); head = head.next
    return out

print("8 node đầu khi quên cắt:", first_k(reverse_bug_cycle(build_list([1, 2, 3]))))

8 node đầu khi quên cắt: [3, 2, 1, 2, 1, 2, 1, 2]


### Lỗi 4 — Khởi tạo `prev = head`

Khi đó head cũ trỏ về chính nó (`1.next = 1`) → chu trình. `prev` phải bắt đầu là `None`, vì đuôi của danh sách đảo phải trỏ `None`.

### Lỗi 5 — Nhầm đảo **giá trị** với đảo **node**

Đổi chỗ `val` (dùng list phụ) cũng ra đúng thứ tự in, nhưng các node vẫn nối theo chiều cũ. Bài phỏng vấn thường muốn đảo liên kết; và nhiều biến thể (đảo theo nhóm k) chỉ làm được khi hiểu đảo liên kết.

## 14. Mẹo nhận dạng pattern

- "Reverse", "đảo", "ngược thứ tự" trên linked list → **ba con trỏ `prev, cur, nxt`**.
- Thao tác cần đi "ngược" trên singly linked list (không có `prev` sẵn) → tự giữ `prev`, hoặc đảo một nửa.
- Kỹ thuật này là **mảnh ghép** của nhiều bài lớn hơn:
  - 234 Palindrome Linked List: tìm giữa (fast/slow) + đảo nửa sau + so sánh.
  - 92 Reverse Linked List II: đảo đoạn `[left, right]`.
  - 25 Reverse Nodes in k-Group: đảo từng nhóm `k` node.
  - 143 Reorder List: tìm giữa + đảo nửa sau + trộn.

## 15. Biến thể

### 15.1. Đảo đoạn từ vị trí `left` đến `right` (LeetCode 92, 1-indexed)
Dùng dummy để có node đứng trước đoạn; đảo đoạn bằng kỹ thuật **chèn lên đầu** (head insertion): lần lượt nhấc node ngay sau `start` lên đứng ngay sau `before`.

### 15.2. Kiểm tra palindrome linked list (LeetCode 234)
Đảo nửa sau rồi so sánh với nửa đầu.

In [20]:
def reverse_between(head, left, right):
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):        # đi tới node đứng trước đoạn
        before = before.next
    start = before.next              # node đầu đoạn, sẽ thành cuối đoạn
    for _ in range(right - left):    # nhấc node sau start lên đầu đoạn
        moving = start.next
        start.next = moving.next
        moving.next = before.next
        before.next = moving
    return dummy.next

print("92:", show(reverse_between(build_list([1, 2, 3, 4, 5]), 2, 4)))   # 1 4 3 2 5

def is_palindrome_list(head):
    slow = fast = head
    while fast and fast.next:        # slow tới giữa
        slow, fast = slow.next, fast.next.next
    second = reverse_iterative(slow)
    p, q = head, second
    while q:
        if p.val != q.val:
            return False
        p, q = p.next, q.next
    return True

print("234:", is_palindrome_list(build_list([1, 2, 2, 1])), is_palindrome_list(build_list([1, 2, 3])))

92: 1 -> 4 -> 3 -> 2 -> 5 -> None
234: True False


## 16. Bài tập tự luyện

### Bài 1 — Vẽ tay
Vẽ trạng thái `prev`, `cur`, `nxt` sau mỗi vòng khi đảo `10 → 20 → 30 → 40`.

### Bài 2 — Dự đoán
Nếu đổi thứ tự hai dòng `prev = cur` và `cur = nxt` thành `cur = nxt` rồi `prev = cur`, kết quả là gì?

### Bài 3 — TODO: tự viết bản lặp

In [21]:
def exercise_reverse(head):
    # TODO: prev = None, cur = head; lưu - đảo - tiến - tiến; trả prev
    pass

# assert to_list(exercise_reverse(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
# assert exercise_reverse(None) is None

### Bài 4 — TODO: tự viết bản đệ quy

In [22]:
def exercise_reverse_recursive(head):
    # TODO: cơ sở; đảo phần sau; nối head vào cuối; cắt head.next
    pass

# assert to_list(exercise_reverse_recursive(build_list([1, 2, 3]))) == [3, 2, 1]

### Bài 5 — Biến thể: đảo theo cặp đầu tiên `k` node
Viết `reverse_first_k(head, k)`: đảo `k` node đầu, phần còn lại giữ nguyên. `k` không vượt quá số node.
Ví dụ `1→2→3→4→5`, `k = 3` → `3→2→1→4→5`.

In [23]:
def exercise_reverse_first_k(head, k):
    # TODO: đảo k bước bằng ba con trỏ, rồi nối head cũ vào cur
    pass

# assert to_list(exercise_reverse_first_k(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]

### Bài 6 — Mini challenge: Reverse Nodes in k-Group (LeetCode 25)
Đảo từng nhóm `k` node liên tiếp; nhóm cuối thiếu node thì giữ nguyên.
`1→2→3→4→5`, `k = 2` → `2→1→4→3→5`; `k = 3` → `3→2→1→4→5`.

In [24]:
def exercise_reverse_k_group(head, k):
    # TODO: dùng dummy; mỗi lần kiểm tra còn đủ k node không; đảo nhóm; nối lại
    pass

# assert to_list(exercise_reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
# assert to_list(exercise_reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | 4 vòng, mỗi vòng đảo 1 mũi tên | bảng 4 hàng | vòng 2: `prev=20, cur=30, nxt=30` (sau khi tiến) |
| 2 | `cur = nxt` trước thì `prev` nhận gì? | `prev = cur = nxt` | `prev` luôn bằng `cur` → mất phần đã đảo |
| 3 | lưu – đảo – tiến – tiến | `nxt = cur.next` | `cur.next = prev; prev = cur; cur = nxt` |
| 4 | Tin lời gọi con | `new_head = f(head.next)` | `head.next.next = head; head.next = None` |
| 5 | Lặp đúng `k` lần | `for _ in range(k)` | cuối cùng `head.next = cur`, trả `prev` |
| 6 | Kết hợp bài 5 nhiều lần | `group_prev` đứng trước nhóm | đếm `k` node bằng con trỏ thăm dò trước khi đảo |

## Đáp án tham khảo

**Bài 1:**

| Vòng | `cur` (đầu vòng) | `nxt` | sau đảo | `prev` | `cur` mới |
|---:|:-:|:-:|---|:-:|:-:|
| 1 | 10 | 20 | `10 → None` | 10 | 20 |
| 2 | 20 | 30 | `20 → 10` | 20 | 30 |
| 3 | 30 | 40 | `30 → 20` | 30 | 40 |
| 4 | 40 | None | `40 → 30` | 40 | None |

**Bài 2:** `cur = nxt` rồi `prev = cur` làm `prev` = node **chưa đảo** tiếp theo, mất tham chiếu tới phần đã đảo. Hơn nữa từ vòng 2, `cur.next = prev` lại gán node vào **chính nó** (`20.next = 20`). Cell dưới kiểm chứng kết quả cuối cùng.

In [25]:
def answer_reverse(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev

def answer_reverse_recursive(head):
    if head is None or head.next is None:
        return head
    new_head = answer_reverse_recursive(head.next)
    head.next.next = head
    head.next = None
    return new_head

def answer_reverse_first_k(head, k):
    prev, cur = None, head
    for _ in range(k):
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
    head.next = cur            # head cũ là đuôi của nhóm, nối vào phần còn lại
    return prev

def answer_reverse_k_group(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        probe = group_prev
        for _ in range(k):                 # kiểm tra còn đủ k node
            probe = probe.next
            if probe is None:
                return dummy.next
        first = group_prev.next
        group_prev.next = answer_reverse_first_k(first, k)
        group_prev = first                 # first giờ là đuôi nhóm

def bai2_wrong(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        cur = nxt
        prev = cur
    return prev

print("Bài 2 kiểm chứng:", show(bai2_wrong(build_list([10, 20, 30, 40]))))
assert bai2_wrong(build_list([10, 20, 30, 40])) is None
assert to_list(answer_reverse(build_list([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert answer_reverse(None) is None
assert to_list(answer_reverse_recursive(build_list([1, 2, 3]))) == [3, 2, 1]
assert to_list(answer_reverse_first_k(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
assert to_list(answer_reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(answer_reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
assert to_list(answer_reverse_k_group(build_list([1, 2, 3, 4]), 4)) == [4, 3, 2, 1]
print("Đáp án tham khảo PASS toàn bộ!")

Bài 2 kiểm chứng: None
Đáp án tham khảo PASS toàn bộ!


Kết quả bài 2 trả `None`: vì `prev = cur` ngay sau `cur = nxt`, đến vòng cuối `cur = None` nên `prev = None` — toàn bộ phần đã đảo bị mất tham chiếu.

## Template nhớ nhanh + checklist submit

```python
prev, cur = None, head
while cur:
    nxt = cur.next      # LƯU
    cur.next = prev     # ĐẢO
    prev = cur          # TIẾN prev
    cur = nxt           # TIẾN cur
return prev             # head mới
```

```python
# Đệ quy
if not head or not head.next: return head
new_head = f(head.next)
head.next.next = head
head.next = None
return new_head
```

**Checklist:**
- [ ] `prev` bắt đầu là `None`.
- [ ] Lưu `nxt` **trước** khi sửa `cur.next`.
- [ ] Trả `prev`, không trả `head`/`cur`.
- [ ] Đệ quy: có `head.next = None`; lưu ý độ sâu 5000.
- [ ] Đã thử rỗng, 1 node, 2 node.